# ĐỒ ÁN DEEP LEARNING: MEDICAL IMAGE SEGMENTATION WITH LIMITED TRAINING DATA
## Đề tài: Phân vùng Phổi trên ảnh X-quang với lượng dữ liệu huấn luyện hạn chế

* **Lĩnh vực:** Computer Vision / Medical AI
* **Đối tượng giải phẫu:** Hai lá phổi (Lungs) trên ảnh X-quang lồng ngực (CXR)
* **Dataset:** Montgomery County & Shenzhen Hospital (NIH - 704 cặp ảnh và mask)
* **Mục tiêu nghiên cứu:** Khảo sát sự suy giảm hiệu năng giữa mô hình huấn luyện từ đầu (Vanilla U-Net) và mô hình tiền huấn luyện (Pretrained U-Net ResNet-34) khi giảm kích thước tập dữ liệu huấn luyện (5%, 10%, 25%, 50%, 100%).


### 1. Kiểm tra cấu hình phần cứng GPU trên Google Colab


In [ ]:
!nvidia-smi


### 2. Cài đặt các thư viện Deep Learning cần thiết


In [ ]:
!pip install -q segmentation-models-pytorch albumentations


### 3. Tải và giải nén bộ dữ liệu X-Ray Lung Segmentation
> **Hướng dẫn:** Chạy ô dưới đây và tải file kaggle.json (lấy từ tài khoản Kaggle của bạn: *Kaggle -> Settings -> Create New Token*). Dữ liệu sẽ được tải trực tiếp về Colab với tốc độ cao (~50MB/s) trong khoảng 30 giây.


In [ ]:
import os
# Cau hinh Kaggle API Token truc tiep tu tai khoan cua ban
os.environ["KAGGLE_API_TOKEN"] = "KGAT_f7482e190c33e3602fd6384906ccacbc"

# Tai va giai nen dataset tu Kaggle voi toc do cao (~50MB/s)
if not os.path.exists("Lung Segmentation"):
    print(">>> Dang tai dataset tu Kaggle...")
    !kaggle datasets download -d nikhilpandey360/chest-xray-masks-and-labels --unzip
    print(">>> Giai nen dataset thanh cong!")
else:
    print(">>> Dataset da ton tai tren Colab!")


### 4. Quét và Khởi tạo các tập phân chia dữ liệu (Data Splits)
* Cắt cố định **20% (140 ảnh)** làm tập Test độc lập.
* **80% còn lại** làm tập Train Pool và trích xuất các mốc: **5% (24 ảnh), 10% (48 ảnh), 25% (120 ảnh), 50% (240 ảnh), 100% (480 ảnh)**.


In [ ]:
import os
import random
import pandas as pd
from pathlib import Path

def prepare_splits(data_dir='Lung Segmentation', output_dir='splits', test_ratio=0.20, seed=42):
    os.makedirs(output_dir, exist_ok=True)
    random.seed(seed)

    cxr_dir = os.path.join(data_dir, 'CXR_png')
    mask_dir = os.path.join(data_dir, 'masks')
    mask_files = set(os.listdir(mask_dir))

    pairs = []
    for f in os.listdir(cxr_dir):
        if not f.endswith('.png'):
            continue
        dataset_type = 'unknown'
        mask_name = None
        if f.startswith('MCUCXR') and f in mask_files:
            dataset_type = 'Montgomery'
            mask_name = f
        elif f.startswith('CHNCXR'):
            target_mask = f.replace('.png', '_mask.png')
            if target_mask in mask_files:
                dataset_type = 'Shenzhen'
                mask_name = target_mask

        if mask_name is not None:
            pairs.append({
                'filename': f,
                'dataset': dataset_type,
                'image_path': os.path.join(cxr_dir, f),
                'mask_path': os.path.join(mask_dir, mask_name)
            })

    mcu = [p for p in pairs if p['dataset'] == 'Montgomery']
    chn = [p for p in pairs if p['dataset'] == 'Shenzhen']
    random.shuffle(mcu)
    random.shuffle(chn)

    n_test_mcu = int(len(mcu) * test_ratio)
    n_test_chn = int(len(chn) * test_ratio)

    test_pool = mcu[:n_test_mcu] + chn[:n_test_chn]
    train_val_pool = mcu[n_test_mcu:] + chn[n_test_chn:]
    random.shuffle(test_pool)
    random.shuffle(train_val_pool)

    n_val = int(len(train_val_pool) * 0.15)
    val_pool = train_val_pool[:n_val]
    train_full = train_val_pool[n_val:]

    pd.DataFrame(test_pool).to_csv(os.path.join(output_dir, 'test_fixed_20pct.csv'), index=False)
    pd.DataFrame(val_pool).to_csv(os.path.join(output_dir, 'val_fixed.csv'), index=False)

    shuffled_train = list(train_full)
    random.shuffle(shuffled_train)

    ratios = {
        'train_5pct.csv': 0.05,
        'train_10pct.csv': 0.10,
        'train_25pct.csv': 0.25,
        'train_50pct.csv': 0.50,
        'train_100pct.csv': 1.00
    }
    for filename, ratio in ratios.items():
        n = max(2, int(len(shuffled_train) * ratio))
        pd.DataFrame(shuffled_train[:n]).to_csv(os.path.join(output_dir, filename), index=False)

    print(f'Tong so cap hop le: {len(pairs)}')
    print(f'Test: {len(test_pool)} | Val: {len(val_pool)} | Train 100%: {len(train_full)}')
    for f, r in ratios.items():
        print(f'  -> {f}: {max(2, int(len(shuffled_train) * r))} mau')

prepare_splits()


### 5. Xây dựng PyTorch Dataset & Pipeline Tăng cường dữ liệu (Augmentation)


In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_transforms(img_size=256, is_train=True):
    if is_train:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.Affine(scale=(0.9, 1.1), rotate=(-15, 15), translate_percent=(-0.0625, 0.0625), p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.3),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

class LungDataset(Dataset):
    def __init__(self, csv_file, img_size=256, is_train=True):
        self.df = pd.read_csv(csv_file)
        self.transforms = get_transforms(img_size=img_size, is_train=is_train)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(row['mask_path'], cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)

        augmented = self.transforms(image=image, mask=mask)
        image = augmented['image']
        mask = augmented['mask']
        if mask.dim() == 2:
            mask = mask.unsqueeze(0)

        return image, mask, row['filename']

def get_loader(csv_file, img_size=256, batch_size=8, is_train=True):
    ds = LungDataset(csv_file, img_size=img_size, is_train=is_train)
    return DataLoader(ds, batch_size=batch_size, shuffle=is_train, num_workers=2, pin_memory=True)


### 6. Kiến trúc Mô hình (Vanilla U-Net vs Pretrained U-Net) & Hàm Mất Mát


In [ ]:
import torch.nn as nn
import segmentation_models_pytorch as smp

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class VanillaUNet(nn.Module):
    def __init__(self, in_channels=3, classes=1, base=32):
        super().__init__()
        self.inc = DoubleConv(in_channels, base)
        self.d1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(base, base*2))
        self.d2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(base*2, base*4))
        self.d3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(base*4, base*8))
        self.d4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(base*8, base*16))

        self.up1 = nn.ConvTranspose2d(base*16, base*8, 2, stride=2)
        self.c1 = DoubleConv(base*16, base*8)
        self.up2 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.c2 = DoubleConv(base*8, base*4)
        self.up3 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.c3 = DoubleConv(base*4, base*2)
        self.up4 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.c4 = DoubleConv(base*2, base)
        self.outc = nn.Conv2d(base, classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.d1(x1)
        x3 = self.d2(x2)
        x4 = self.d3(x3)
        x5 = self.d4(x4)
        x = self.c1(torch.cat([x4, self.up1(x5)], dim=1))
        x = self.c2(torch.cat([x3, self.up2(x)], dim=1))
        x = self.c3(torch.cat([x2, self.up3(x)], dim=1))
        x = self.c4(torch.cat([x1, self.up4(x)], dim=1))
        return self.outc(x)

def build_model(model_name='unet', encoder='resnet34'):
    if model_name == 'vanilla_unet':
        return VanillaUNet(in_channels=3, classes=1)
    return smp.Unet(encoder_name=encoder, encoder_weights='imagenet', in_channels=3, classes=1)

class ComboLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        bce = self.bce(logits, targets)
        p = torch.sigmoid(logits).view(-1)
        t = targets.view(-1)
        inter = (p * t).sum()
        dice = (2. * inter + 1e-6) / (p.sum() + t.sum() + 1e-6)
        return 0.5 * bce + 0.5 * (1. - dice)

@torch.no_grad()
def compute_dice_iou(logits, targets):
    preds = (torch.sigmoid(logits) > 0.5).float().view(logits.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    inter = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1)
    dice = (2. * inter + 1e-6) / (union + 1e-6)
    iou = (inter + 1e-6) / (union - inter + 1e-6)
    return dice.mean().item(), iou.mean().item()


### 7. Engine Huấn luyện & Đánh giá


In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_and_eval(model_name, encoder, ratio_pct, epochs=25, batch_size=8, lr=3e-4):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_loader = get_loader(f'splits/train_{ratio_pct}pct.csv', batch_size=batch_size, is_train=True)
    val_loader = get_loader('splits/val_fixed.csv', batch_size=batch_size, is_train=False)
    test_loader = get_loader('splits/test_fixed_20pct.csv', batch_size=batch_size, is_train=False)

    model = build_model(model_name, encoder).to(device)
    criterion = ComboLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_dice = 0.0
    best_weights = None

    for epoch in range(1, epochs + 1):
        model.train()
        for imgs, masks, _ in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), masks)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        dices = []
        with torch.no_grad():
            for imgs, masks, _ in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                d, _ = compute_dice_iou(model(imgs), masks)
                dices.append(d)
        val_dice = sum(dices) / len(dices)
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            best_weights = model.state_dict().copy()

    # Danh gia tren Test set doc lap
    model.load_state_dict(best_weights)
    model.eval()
    test_dices, test_ious = [], []
    with torch.no_grad():
        for imgs, masks, _ in test_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            d, i = compute_dice_iou(model(imgs), masks)
            test_dices.append(d)
            test_ious.append(i)

    test_dice = sum(test_dices) / len(test_dices)
    test_iou = sum(test_ious) / len(test_ious)
    print(f'[{model_name.upper()} | Ratio: {ratio_pct:3d}%] -> Test Dice: {test_dice*100:.2f}% | Test IoU: {test_iou*100:.2f}%')
    return test_dice, test_iou


### 8. Khởi chạy toàn bộ Benchmark Data Scaling (\%, 10\%, 25\%, 50\%, 100\%$)
> Quá trình này sẽ đo lường hiệu năng của **Vanilla U-Net (From scratch)** và **Pretrained U-Net (ResNet-34)** trên từng tỷ lệ dữ liệu.


In [ ]:
ratios = [5, 10, 25, 50, 100]
results = []

print('=== 1. HUAN LUYEN PRETRAINED U-NET (RESNET-34) ===')
for r in ratios:
    dice, iou = train_and_eval('unet', 'resnet34', ratio_pct=r, epochs=25, batch_size=8)
    results.append({'model': 'Pretrained U-Net (ResNet-34)', 'ratio': r, 'test_dice': dice, 'test_iou': iou})

print('\n=== 2. HUAN LUYEN VANILLA U-NET (FROM SCRATCH) ===')
for r in ratios:
    dice, iou = train_and_eval('vanilla_unet', 'none', ratio_pct=r, epochs=25, batch_size=8)
    results.append({'model': 'Vanilla U-Net (From Scratch)', 'ratio': r, 'test_dice': dice, 'test_iou': iou})

df_res = pd.DataFrame(results)
df_res.to_csv('data_scaling_benchmark_results.csv', index=False)
display(df_res)


### 9. Vẽ Đồ thị Khoa học: Data Scaling Curves & Trực quan hóa
Đồ thị so sánh trực quan minh chứng luận điểm khoa học của đề tài: *Mô hình Pretrained duy trì phong độ vượt trội khi dữ liệu giảm sâu.*


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
for m_name, group in df_res.groupby('model'):
    plt.plot(group['ratio'], group['test_dice'] * 100, marker='o', linewidth=2.5, markersize=8, label=m_name)

plt.title('Khảo sát tương quan: Tỷ lệ dữ liệu huấn luyện vs. Dice Score trên tập Test', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Tỷ lệ dữ liệu huấn luyện (%)', fontsize=12)
plt.ylabel('Test Dice Score (%)', fontsize=12)
plt.xticks([5, 10, 25, 50, 100], ['5% (24 mẫu)', '10% (48 mẫu)', '25% (120 mẫu)', '50% (240 mẫu)', '100% (480 mẫu)'])
plt.ylim(0, 100)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('data_scaling_comparison.png', dpi=300)
plt.show()
